# station_data.db is the database


I have to load the dataframe and start using SQL.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('../src')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

from loaders import *

In [ ]:
df = load_range(2000,1,2023,2)
df.head()

In [ ]:
import sqlite3

db_path = "../data/station_data.db"
connection = sqlite3.connect(db_path)
cursor = connection.cursor()

https://sqlbolt.com/lesson/creating_tables

I create the table:

In [ ]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS weather(
        date TEXT PRIMARY KEY,
        temp REAL, 
        humidity REAL,
        vapor_pressure REAL,
        vapor_deficit REAL, 
        dew_point REAL,
        rain_mm REAL, 
        radiation REAL, 
        radiation_accum REAL
    );
""")

connection.commit()

In [ ]:
cursor.execute("SELECT name FROM sqlite_master WHERE type = 'table';")
cursor.fetchall()

In [ ]:
# df.to_sql("weather", con = connection, if_exists="append", index=False, chunksize = 10_000)

The table is created.

In [ ]:
cursor.execute("SELECT COUNT(*) FROM weather;")
cursor.fetchone()

# Consulting data with SQL

## Answering questions with SQL

### Selecting the first 5 datapoints

In [ ]:
query = """SELECT * FROM weather LIMIT 5;"""

pd.read_sql_query(query,connection)

### What data was recorded on january 1, 2000?

In [ ]:
query = """SELECT * FROM weather WHERE date<'2000-01-02' AND date>='2000-01-01'"""

pd.read_sql_query(query,connection)

### What was the average temperature on January 1, 2000?
Answer: 2.922 ºC

In [ ]:
query = """SELECT AVG(temp) FROM weather WHERE date<'2000-01-02' AND date>='2000-01-01'"""

pd.read_sql_query(query, connection)

### What were the minimim and maximum temperatures on March 21, 2007? (firend's bithday)

In [ ]:
query = """SELECT MAX(temp) AS max_temp ,MIN(temp) AS min_temp FROM weather WHERE date<'2007-03-22' AND date>='2007-03-21'"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

In [ ]:
query = """SELECT MAX(temp) AS max_temp ,MIN(temp) AS min_temp FROM weather WHERE date>='2010-07-02' AND date<'2010-07-03'"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### How did temperature change on November 28, 2007? (friend's birthday)

In [ ]:
query = """SELECT strftime('%H:%M',date) AS time,temp FROM weather WHERE date>='2007-11-28' AND date<'2007-11-29'"""

temp_daniel = pd.read_sql_query(query, connection)

temp_daniel.style.hide(axis="index")

In [ ]:
plt.figure(dpi = 200)
plt.plot(temp_daniel["time"],temp_daniel["temp"], color = "red")
plt.xticks(temp_daniel["time"][::12],rotation = 45)
plt.axvline(x = "06:00", color = "grey", label = "Daniel", linestyle = ":")
plt.axvspan("00:00", "06:00", color="gray", alpha=0.12)


plt.xlabel("Hora")
plt.ylabel("Temperatura ºC")
plt.title("Temperatura el 28 Nov 2007 - Salamanca")
plt.show()

### How many observations were recorded each year? 

In [ ]:
query = """SELECT strftime('%Y',date) AS year, COUNT(*) AS number_of_observations FROM weather GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

### What was the average temperature for each year?

In [ ]:
query = """SELECT strftime('%Y',date) AS year, AVG(temp) AS avg_temp FROM weather WHERE date < '2023-01-01' GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")

In [ ]:
plt.figure(dpi = 200)

plt.plot(result["year"], result["avg_temp"], color = "red")
plt.xticks(rotation = 45)
plt.xlabel("year")
plt.ylabel("avg_temp ºC")
plt.title("Average temperature 2000-2022")
plt.grid()
plt.tight_layout()

### How did missing temperature values change by year?

In [ ]:
query = """SELECT strftime('%Y',date) AS year, COUNT(*)-COUNT(temp) AS missing_temp FROM weather GROUP BY year ORDER BY year"""

result = pd.read_sql_query(query, connection)

result.style.hide(axis="index")